# BPA Master Workflow: Pattern Classification (Task 1) & TSD Estimation (Task 2)
**Toronto Metropolitan University — Major Research Project (MRP)**

---

## 🩸 Project Overview & Scientific Workflow
This master notebook coordinates the end-to-end execution of both core tasks in the Major Research Project:
1. **Task 1: Multi-Task Bloodstain Pattern Classification**: Predicts pattern categories (excluding Cough Spatter to prevent leakage) and physical force mechanisms, leading to physics-based Area of Origin calculations.
2. **Task 2: Time Since Deposition (TSD) Estimation**: Predicts temporal age splits (*Fresh*, *Intermediate*, *Aged*) based on hemoglobin oxidation color-space shifts over a 28-day drying cycle.

---

## ⚙️ Phase 1: Environment Setup & GPU Verification
Installs required libraries and checks if a high-speed GPU (T4 on Google Colab) is active.

In [ ]:
# Check GPU acceleration
import torch
print('\n=== GPU Device Check ===')
if torch.cuda.is_available():
    print(f'[+] GPU Active: {torch.cuda.get_device_name(0)}')
else:
    print('[!] GPU not active. Running training from scratch is highly recommended with GPU support.')

In [ ]:
# Install required dependencies
!pip install -q timm scikit-learn seaborn matplotlib python-docx albumentations jinja2
print('[+] Dependency libraries installed successfully!')

--- 
## 🎯 Option A: Run Inference Using Saved Models
If you want to immediately run evaluations or test individual images using the pre-trained weights already saved in the repository, use this section. No training from scratch is required.

### Step A.1: Navigate to Repository Root
To run imports and load weight files, Colab needs to be pointing to the repository's root folder. Choose one of the options below:

In [ ]:
import os
import sys

# =====================================================================
# METHOD 1: Clone Directly to Colab Local VM (Easiest for anyone)
# =====================================================================
# Uncomment the two lines below to clone and cd to the repo root:
# !git clone https://github.com/shahidabatool/A-Hybrid-Framework-for-Forensic-Bloodstain-Pattern-Analysis.git
# %cd A-Hybrid-Framework-for-Forensic-Bloodstain-Pattern-Analysis

# =====================================================================
# METHOD 2: Mount Personal Google Drive (Recommended for persistent development)
# =====================================================================
# First, open the Shared Drive Link and select 'Add shortcut to Drive' on the 'MRP' folder.
# Then, uncomment the three lines below to mount and navigate directly to it:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # %cd /content/drive/MyDrive/MRP
except ImportError:
    print('[+] Running in local terminal environment.')

print(f'[+] Current Working Directory set to: {os.getcwd()}')

### ⚠️ Notice: Pre-trained Weights Download
Because deep learning weight checkpoints (`.pth` files) are large and exceed GitHub's 100MB file limit, they are excluded from the git push commits.

- **If using Method 1 (Git Clone)**: Please download the `.pth` weights from the [Shared Google Drive Archive](https://drive.google.com/drive/folders/1Qg8CjBzJ2wPJIfGgGEfSezEaNndCxuGy?usp=sharing) and upload them manually into the respective local Colab folders (`Task_1_Classification/Models/` and `Task_2_TSD/Models/`).
- **If using Method 2 (Drive Mount)**: Because you added a shortcut to your Drive, all weights and datasets inside `MRP` are already loaded and ready in your directory! No downloads or uploads are required.

### Step A.2: Import Architectures & Load Weights
This dynamically appends the correct model directories relative to your current workspace root folder.

In [ ]:
import sys
import torch
import torchvision.transforms as transforms
from PIL import Image

# Dynamically append import folders relative to current working directory
sys.path.append(os.path.join(os.getcwd(), 'Task_1_Classification/Code/training'))
sys.path.append(os.path.join(os.getcwd(), 'Task_2_TSD/Code/models'))

from train_efficientnet import BPAMultiTaskEfficientNet
from bloodnet import bloodnet50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[+] Using device: {device}')

# 1. Load Task 1 Classifier Model (4-class pattern setup excluding Cough Spatter to prevent leakage)
t1_model = BPAMultiTaskEfficientNet(num_pattern=4, num_mechanism=3)
t1_weights_path = 'Task_1_Classification/Models/model1_efficientnet_b0.pth'
if os.path.exists(t1_weights_path):
    t1_model.load_state_dict(torch.load(t1_weights_path, map_location=device))
    t1_model.to(device).eval()
    print('[+] Task 1 (EfficientNet-B0) weights loaded successfully!')
else:
    print(f'[!] Warning: Weights file not found at {t1_weights_path}')

# 2. Load Task 2 TSD Model
t2_model = bloodnet50(num_classes=3)
t2_weights_path = 'Task_2_TSD/Models/best_tsd_model_resnet50.pth'
if os.path.exists(t2_weights_path):
    t2_model.load_state_dict(torch.load(t2_weights_path, map_location=device))
    t2_model.to(device).eval()
    print('[+] Task 2 (ResNet-50 CBAM) weights loaded successfully!')
else:
    print(f'[!] Warning: Weights file not found at {t2_weights_path}')

### Step A.3: Run Unified Inference Example
Evaluate any custom bloodstain image path, printing out the predictions.

In [ ]:
def predict_bloodstain(image_path):
    if not os.path.exists(image_path):
        print(f'[!] File not found: {image_path}')
        return
        
    # Preprocessing transforms
    t1_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    t2_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    img = Image.open(image_path).convert('RGB')
    
    # Task 1 Classification
    img_t1 = t1_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        pattern_out, mech_out = t1_model(img_t1)
        pattern_idx = torch.argmax(pattern_out, dim=1).item()
        mech_idx = torch.argmax(mech_out, dim=1).item()
        
    # Sourced active 4-class labels matching training output layer
    t1_pattern_labels = ['Gunshot', 'Impact Spatter', 'Passive Drip', 'Transfer/Wipe']
    t1_mech_labels = ['Passive', 'Low Velocity', 'Medium/High Velocity']
    
    # Task 2 Temporal Age Prediction
    img_t2 = t2_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        age_out = t2_model(img_t2)
        age_idx = torch.argmax(age_out, dim=1).item()
        
    t2_age_labels = ['Fresh (1 Day)', 'Intermediate (7-14 Days)', 'Aged (21-28 Days)']
    
    print('==================================================')
    print(f'🩸 FORENSIC ANALYSIS REPORT FOR: {os.path.basename(image_path)}')
    print('==================================================')
    print(f'[+] Predicted Pattern:    {t1_pattern_labels[pattern_idx]}')
    print(f'[+] Predicted Mechanism:  {t1_mech_labels[mech_idx]}')
    print(f'[+] Estimated Age (TSD):  {t2_age_labels[age_idx]}')
    print('==================================================')

# Test with a placeholder or actual image path:
# predict_bloodstain('Task_1_Classification/Data/Organized/Passive_Drip/sample_drop.jpg')

--- 
## ⚙️ Option B: Re-run Training & Evaluation Pipelines
If you want to re-run the entire training cycle (e.g. on Colab virtual disk for faster speeds), follow this section.

### Step B.1: Extract Datasets from Google Drive
Extracts datasets to local high-speed VM storage to maximize training speed. Make sure you mount your Drive and set paths accordingly.

In [ ]:
import os, shutil

# Update these path variables to point to where the ZIP archives are uploaded in your Drive:
task1_zip_drive = '/content/drive/MyDrive/Task_1_Classification_Colab.zip'
task2_zip_drive = '/content/drive/MyDrive/Task_2_TSD_Colab.zip'

local_extract_dir = '/content/Task_1'
if os.path.exists(task1_zip_drive):
    print('[+] Extracting Task 1 datasets to local VM...')
    shutil.unpack_archive(task1_zip_drive, local_extract_dir)
    print('[+] Symlinking /content/Code for Task 1 scripts.')
    if os.path.exists('/content/Code'): os.remove('/content/Code')
    os.symlink(os.path.join(local_extract_dir, 'Code'), '/content/Code')
else:
    print(f'[!] Task 1 zip archive not found at {task1_zip_drive}. Verify path.')

local_tsd_dir = '/content/Task_2'
if os.path.exists(task2_zip_drive):
    print('[+] Extracting Task 2 datasets to local VM...')
    shutil.unpack_archive(task2_zip_drive, local_tsd_dir)
    if os.path.exists('/content/TSD_Code'): os.remove('/content/TSD_Code')
    os.symlink(os.path.join(local_tsd_dir, 'Code'), '/content/TSD_Code')
else:
    print(f'[!] Task 2 zip archive not found at {task2_zip_drive}. Verify path.')

### Step B.2: Run Parent-Image Leakage Audit (Task 1)

In [ ]:
!python /content/Code/audit_data_leakage.py

### Step B.3: Train Task 1 Convolutional Models

In [ ]:
print('=== Training Multi-Task EfficientNet-B0 ===')
!python /content/Code/train_efficientnet.py --data_dir /content/Task_1/Data/Augmented/train --val_dir /content/Task_1/Data/Augmented/val --epochs 10 --batch_size 32

print('\n=== Training Multi-Task ResNet-50 ===')
!python /content/Code/train_resnet50.py --data_dir /content/Task_1/Data/Augmented/train --val_dir /content/Task_1/Data/Augmented/val --epochs 10 --batch_size 32

print('\n=== Training Multi-Task ConvNeXt-Tiny ===')
!python /content/Code/train_convnext.py

### Step B.4: Evaluate Task 1 Classifiers

In [ ]:
print('=== Evaluating EfficientNet-B0 ===')
!python /content/Code/evaluate_efficientnet.py --test_dir /content/Task_1/Data/Augmented/test

print('\n=== Evaluating ResNet-50 ===')
!python /content/Code/evaluate_resnet50.py --test_dir /content/Task_1/Data/Augmented/test

print('\n=== Evaluating ConvNeXt-Tiny ===')
!python /content/Code/evaluate_convnext.py

### Step B.5: Train Task 2 (TSD) Models

In [ ]:
print('=== Training Fine-Tuned ResNet-50 TSD ===')
!python /content/TSD_Code/train_tsd_resnet50.py \
  --data_dir /content/Task_2/Text/BloodNet_50k_Images/data/train1 \
  --val_dir /content/Task_2/Text/BloodNet_50k_Images/data/test \
  --weights_path /content/Task_2/Text/BloodNet_50k_Images/bloodnet50_new.pth \
  --output_dir /content/Task_2/Models \
  --epochs 10 \
  --batch_size 128 \
  --lr 1e-4

print('\n=== Training EfficientNet-B0 TSD ===')
!python /content/TSD_Code/train_tsd_efficientnet.py \
  --data_dir /content/Task_2/Text/BloodNet_50k_Images/data/train1 \
  --val_dir /content/Task_2/Text/BloodNet_50k_Images/data/test \
  --output_dir /content/Task_2/Models \
  --epochs 10 \
  --batch_size 128 \
  --lr 1e-4

### Step B.6: Evaluate Task 2 TSD Backbones

In [ ]:
!python /content/TSD_Code/evaluate_tsd.py \
  --test_dir /content/Task_2/Text/BloodNet_50k_Images/data/outside_test \
  --baseline_weights /content/Task_2/Text/BloodNet_50k_Images/bloodnet50_new.pth \
  --resnet50_weights /content/Task_2/Models/best_tsd_model_resnet50.pth \
  --efficientnet_weights /content/Task_2/Models/best_tsd_model_efficientnet.pth \
  --convnext_weights /content/Task_2/Models/best_tsd_model_convnext.pth \
  --output_dir /content/Task_2/Evaluation

### Step B.7: Sync Checkpoints and Reports Back to Drive

In [ ]:
!python /content/Code/generate_word_report.py
!python /content/TSD_Code/generate_tsd_word_report.py

# Sync Task 1
!mkdir -p /content/drive/MyDrive/Task_1_Classification/Evaluation
!cp -r /content/Task_1/Evaluation/* /content/drive/MyDrive/Task_1_Classification/Evaluation/

# Sync Task 2
!mkdir -p /content/drive/MyDrive/Task_2_TSD/Evaluation
!mkdir -p /content/drive/MyDrive/Task_2_TSD/Models
!cp -r /content/Task_2/Evaluation/* /content/drive/MyDrive/Task_2_TSD/Evaluation/
!cp -r /content/Task_2/Models/* /content/drive/MyDrive/Task_2_TSD/Models/

print('[+] All checkpoints and Word reports synced to Google Drive successfully!')

--- 
## 💻 Phase 4: Run the Streamlit Dashboard (Local Execution)
To start the visual analytics interface on your local Mac/Windows machine, run these command line prompts:

```bash
# 1. Navigate to the project root directory
cd /path/to/cloned/repository/MRP

# 2. Activate the virtual environment
source mrp_env/bin/activate

# 3. Run the application server
streamlit run dashboard.py
```
This will launch the web application local instance at `http://localhost:8501`, loading models dynamically from the respective models folders.